# Playwright UI Test: Import and Display TTPIBA-NR.json

This notebook uses Playwright to simulate a user importing the provided TTPIBA-NR.json roster file and verifies that the roster is loaded and visible in the UI as expected.

## 1. Install and Import Playwright

Install Playwright if not already installed, and import the required modules for browser automation.

In [ ]:
# Install Playwright if needed
!pip install playwright > /dev/null
!playwright install chromium > /dev/null

from playwright.sync_api import sync_playwright
import json
import os

## 2. Load and Parse the Roster JSON

Read the TTPIBA-NR.json file and parse its contents into a Python dictionary.

In [ ]:
# Load the TTPIBA-NR.json file
json_path = '../TTPIBA-NR.json' if not os.path.exists('TTPIBA-NR.json') else 'TTPIBA-NR.json'
with open(json_path, 'r') as f:
    roster_data = json.load(f)

# Preview the loaded roster name
display_name = roster_data['roster'].get('name', 'UNKNOWN')
display_name

## 3. Launch Browser and Navigate to Test Page

Use Playwright to launch a browser instance and navigate to the local web app for testing.

In [ ]:
# Start Playwright and open the local app
playwright = sync_playwright().start()
browser = playwright.chromium.launch(headless=True)
context = browser.new_context()
page = context.new_page()

# Adjust the URL if your local server runs on a different port or path
page.goto('http://localhost:8000/')

## 4. Inject Roster Data into the Page

Simulate the user importing the TTPIBA-NR.json file using the Import Roster JSON button and file input.

In [ ]:
# Simulate user clicking Import and uploading the JSON file
import time

# Click the Import button to show the file input
page.click('#import-roster-btn')

# Upload the TTPIBA-NR.json file
file_input = page.query_selector('input[type="file"]')
file_input.set_input_files(json_path)

# Wait for the UI to process the import (adjust selector as needed)
time.sleep(1)

## 5. Simulate User Actions

Verify that the roster is loaded and visible in the UI, and interact with the page as a user would.

In [ ]:
# Wait for the roster name to appear in the UI (adjust selector as needed)
from playwright.sync_api import TimeoutError

try:
    page.wait_for_selector('.roster-name', timeout=3000)
    roster_name_ui = page.inner_text('.roster-name')
except TimeoutError:
    roster_name_ui = None

print('Roster name in UI:', roster_name_ui)

## 6. Capture and Assert User-visible Output

Capture a screenshot and assert that the roster name and at least one unit are visible and not 'UNKNOWN'.

In [ ]:
# Take a screenshot for visual confirmation
screenshot_path = 'imported_roster_ui.png'
page.screenshot(path=screenshot_path)

# Assert the roster name is not UNKNOWN and matches the imported data
assert roster_name_ui and roster_name_ui != 'UNKNOWN', f"Roster name is not visible or is UNKNOWN: {roster_name_ui}"
assert display_name in roster_name_ui, f"Expected roster name '{display_name}' not found in UI: {roster_name_ui}"

# Check for at least one unit name from the imported data
unit_names = [sel['name'] for force in roster_data['roster']['forces'] for sel in force['selections'] if 'name' in sel]
found_unit = False
for unit in unit_names:
    if unit in page.content():
        found_unit = True
        break
assert found_unit, "No unit names from the imported roster are visible in the UI."

print('UI test passed. Screenshot saved:', screenshot_path)

## 7. Close Browser

Cleanly close the browser instance after the test completes.

In [ ]:
# Clean up
context.close()
browser.close()
playwright.stop()